# Mobility Feature Engineering

## Project Title: Traffic Accident Risk Prediction (TARP)

**Unit:** SIT782  
**Prepared by:** Subathira Thinakaran  

**Project Team:**  
- Suba (225094537)  
- Burhan (224802775)  
- Khalid (224696667)  

**Task:** Feature Engineering – Mobility Features (Week 4 of 8)

## Objective
This notebook creates mobility-based features from the processed pedestrian and bicycle datasets for use in later dataset integration and predictive modelling.

## Output

The feature engineering process generated a unified mobility feature dataset:

- mobility_features.csv

This dataset includes:
- hour_of_day  
- day_of_week  
- weekend_flag  
- pedestrian_density  
- bicycle_density  

It represents relative mobility intensity across temporal conditions and is ready for integration with crash and weather datasets.

In [23]:
# -----------------------------
# 1. Import libraries
# -----------------------------
from pathlib import Path
import pandas as pd

In [2]:
from google.colab import files
uploaded = files.upload()

Saving pedestrian_clean.csv to pedestrian_clean.csv
Saving bicycle_clean.csv to bicycle_clean.csv


## Load Processed Datasets

This section loads the cleaned pedestrian and bicycle datasets prepared in the previous stages.

In [55]:
# -----------------------------
# 2. Load processed datasets
# -----------------------------
PROCESSED_DATA_DIR = Path("data/processed")

pedestrian_file = PROCESSED_DATA_DIR / "pedestrian_clean.csv"
bicycle_file = PROCESSED_DATA_DIR / "bicycle_clean.csv"

# Fallback for Colab/manual upload
if not pedestrian_file.exists():
    print("Using uploaded files in current directory...")
    pedestrian_file = Path("pedestrian_clean.csv")
    bicycle_file = Path("bicycle_clean.csv")

pedestrian_df = pd.read_csv(pedestrian_file)
bicycle_df = pd.read_csv(bicycle_file)

print("Pedestrian dataset shape:", pedestrian_df.shape)
print("Bicycle dataset shape:", bicycle_df.shape)

Using uploaded files in current directory...
Pedestrian dataset shape: (5000, 11)
Bicycle dataset shape: (3109, 15)


## Prepare Temporal Fields

This step standardises key temporal fields so that the mobility features can align with the crash feature dataset. Shared keys such as `hour_of_day`, `day_of_week`, and `weekend_flag` are created or validated for both pedestrian and bicycle data.

In [56]:
# -----------------------------
# 3. Prepare temporal fields
# -----------------------------
pedestrian_df["timestamp"] = pd.to_datetime(pedestrian_df["timestamp"], errors="coerce")
pedestrian_df["hour_of_day"] = pedestrian_df["timestamp"].dt.hour
pedestrian_df["day_of_week"] = pedestrian_df["timestamp"].dt.day_name()
pedestrian_df["weekend_flag"] = pedestrian_df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

bicycle_df["date"] = pd.to_datetime(bicycle_df["date"], errors="coerce")
bicycle_df["hour_of_day"] = pd.to_numeric(bicycle_df["hour"], errors="coerce")
bicycle_df["weekend_flag"] = bicycle_df["is_weekend"].astype(int)

print("Pedestrian temporal preview:")
display(pedestrian_df[["timestamp", "hour_of_day", "day_of_week", "weekend_flag"]].head())

print("Bicycle temporal preview:")
display(bicycle_df[["date", "hour_of_day", "day_of_week", "weekend_flag"]].head())

# Optional consistency check
derived_bike_day = bicycle_df["date"].dt.day_name()
day_match = (derived_bike_day == bicycle_df["day_of_week"]).all()
print("Bicycle day_of_week matches date:", day_match)

Pedestrian temporal preview:


,timestamp,hour_of_day,day_of_week,weekend_flag
0,2024-07-21 09:00:00,9,Sunday,1
1,2025-06-09 02:00:00,2,Monday,0
2,2025-08-03 05:00:00,5,Sunday,1
3,2024-08-29 20:00:00,20,Thursday,0
4,2026-01-04 20:00:00,20,Sunday,1


Bicycle temporal preview:


,date,hour_of_day,day_of_week,weekend_flag
0,2015-04-24,6,Friday,0
1,2015-04-24,16,Friday,0
2,2015-04-24,19,Friday,0
3,2015-04-25,15,Saturday,1
4,2015-04-25,18,Saturday,1


Bicycle day_of_week matches date: True


### Observation

Temporal fields were successfully prepared for both datasets. The pedestrian dataset now includes hour, weekday, and weekend indicators derived from the full timestamp, while the bicycle dataset has been aligned to the same temporal structure.

This shared format ensures that mobility features can later be merged more consistently with crash features during dataset integration.

## Create Pedestrian Mobility Summary

This step aggregates pedestrian counts across hour of day, day of week, and weekend indicator to create a detailed temporal mobility profile.

In [57]:
# -----------------------------
# 4. Create pedestrian mobility summary
# -----------------------------
weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

ped_summary = (
    pedestrian_df
    .groupby(["hour_of_day", "day_of_week", "weekend_flag"], as_index=False)["pedestriancount"]
    .sum()
)

ped_summary["day_of_week"] = pd.Categorical(
    ped_summary["day_of_week"],
    categories=weekday_order,
    ordered=True
)

ped_summary = ped_summary.sort_values(
    ["day_of_week", "hour_of_day"]
).reset_index(drop=True)

print("Pedestrian mobility summary shape:", ped_summary.shape)
ped_summary.head()

Pedestrian mobility summary shape: (168, 4)


,hour_of_day,day_of_week,weekend_flag,pedestriancount
0,0,Monday,0,1611
1,1,Monday,0,539
2,2,Monday,0,414
3,3,Monday,0,153
4,4,Monday,0,336


### Observation

The pedestrian mobility summary contains 168 records representing all combinations of hourly and weekly temporal conditions.

This structure captures variations in pedestrian activity across both time of day and day of week, providing a more detailed representation of mobility patterns compared to simple hourly aggregation.

## Create Bicycle Mobility Summary

This step aggregates bicycle counts across hour of day, day of week, and weekend indicator to create a detailed temporal mobility profile.

In [58]:
# -----------------------------
# 5. Create bicycle mobility summary
# -----------------------------
weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

bike_summary = (
    bicycle_df
    .groupby(["hour_of_day", "day_of_week", "weekend_flag"], as_index=False)["bike"]
    .sum()
)

bike_summary["day_of_week"] = pd.Categorical(
    bike_summary["day_of_week"],
    categories=weekday_order,
    ordered=True
)

bike_summary = bike_summary.sort_values(
    ["day_of_week", "hour_of_day"]
).reset_index(drop=True)

print("Bicycle mobility summary shape:", bike_summary.shape)
bike_summary.head()

Bicycle mobility summary shape: (168, 4)


,hour_of_day,day_of_week,weekend_flag,bike
0,0,Monday,0,5.0
1,1,Monday,0,0.0
2,2,Monday,0,2.0
3,3,Monday,0,2.0
4,4,Monday,0,8.0


### Observation

The bicycle mobility summary contains 168 records representing all combinations of hourly and weekly temporal conditions.

This structure captures variations in bicycle activity across both time of day and day of week, providing a detailed temporal view of cycling behaviour.

## Create Pedestrian Density Feature

This step creates a relative pedestrian activity feature using min-max normalisation across all temporal conditions.

In [59]:
# -----------------------------
# 6. Create pedestrian density
# -----------------------------
ped_min = ped_summary["pedestriancount"].min()
ped_max = ped_summary["pedestriancount"].max()

if ped_max != ped_min:
    ped_summary["pedestrian_density"] = (
        (ped_summary["pedestriancount"] - ped_min) / (ped_max - ped_min)
    )
else:
    ped_summary["pedestrian_density"] = 0

print("Pedestrian density summary:")
print(ped_summary[["hour_of_day", "day_of_week", "weekend_flag", "pedestrian_density"]].head())

print("\nValidation:")
print("Min density:", ped_summary["pedestrian_density"].min())
print("Max density:", ped_summary["pedestrian_density"].max())

print("\nDensity distribution summary:")
print(ped_summary["pedestrian_density"].describe())

Pedestrian density summary:
   hour_of_day day_of_week  weekend_flag  pedestrian_density
0            0      Monday             0            0.046343
1            1      Monday             0            0.012269
2            2      Monday             0            0.008296
3            3      Monday             0            0.000000
4            4      Monday             0            0.005817

Validation:
Min density: 0.0
Max density: 1.0

Density distribution summary:
count    168.000000
mean       0.341155
std        0.271194
min        0.000000
25%        0.068354
50%        0.334366
75%        0.520883
max        1.000000
Name: pedestrian_density, dtype: float64


### Observation

The pedestrian density feature was successfully created using min-max normalisation. The values range from 0 to 1, where 0 represents the lowest observed pedestrian activity and 1 represents the highest.

By applying normalisation across all temporal combinations, the feature captures variations in pedestrian intensity across different hours and days of the week, providing a richer representation of mobility patterns.

## Create Bicycle Density Feature

This step creates a relative bicycle activity feature using min-max normalisation across all temporal conditions.

In [60]:
# -----------------------------
# 7. Create bicycle density
# -----------------------------
bike_min = bike_summary["bike"].min()
bike_max = bike_summary["bike"].max()

if bike_max != bike_min:
    bike_summary["bicycle_density"] = (
        (bike_summary["bike"] - bike_min) / (bike_max - bike_min)
    )
else:
    bike_summary["bicycle_density"] = 0

print("Bicycle density summary:")
print(bike_summary[["hour_of_day", "day_of_week", "weekend_flag", "bicycle_density"]].head())

print("\nValidation:")
print("Min density:", bike_summary["bicycle_density"].min())
print("Max density:", bike_summary["bicycle_density"].max())

print("\nBicycle density distribution:")
print(bike_summary["bicycle_density"].describe())


Bicycle density summary:
   hour_of_day day_of_week  weekend_flag  bicycle_density
0            0      Monday             0         0.013089
1            1      Monday             0         0.000000
2            2      Monday             0         0.005236
3            3      Monday             0         0.005236
4            4      Monday             0         0.020942

Validation:
Min density: 0.0
Max density: 1.0

Bicycle density distribution:
count    168.000000
mean       0.100785
std        0.118949
min        0.000000
25%        0.022906
50%        0.075916
75%        0.128272
max        1.000000
Name: bicycle_density, dtype: float64


### Observation

The bicycle density feature was successfully created using min-max normalisation. The values range from 0 to 1, where 0 represents the lowest observed bicycle activity and 1 represents the highest.

By applying normalisation across all temporal combinations, the feature captures variations in bicycle intensity across different hours and days of the week, providing a detailed representation of cycling behaviour.

## Select Final Feature Datasets

This step retains only the final engineered mobility features and the temporal keys required for later integration.

In [61]:
# -----------------------------
# 8. Select final feature datasets
# -----------------------------
weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

pedestrian_features = ped_summary[
    ["hour_of_day", "day_of_week", "weekend_flag", "pedestrian_density"]
].copy()

bicycle_features = bike_summary[
    ["hour_of_day", "day_of_week", "weekend_flag", "bicycle_density"]
].copy()

pedestrian_features["day_of_week"] = pd.Categorical(
    pedestrian_features["day_of_week"],
    categories=weekday_order,
    ordered=True
)

bicycle_features["day_of_week"] = pd.Categorical(
    bicycle_features["day_of_week"],
    categories=weekday_order,
    ordered=True
)

pedestrian_features = pedestrian_features.sort_values(
    ["day_of_week", "hour_of_day"]
).reset_index(drop=True)

bicycle_features = bicycle_features.sort_values(
    ["day_of_week", "hour_of_day"]
).reset_index(drop=True)

print("Pedestrian feature shape:", pedestrian_features.shape)
print("Bicycle feature shape:", bicycle_features.shape)

display(pedestrian_features.head())
display(bicycle_features.head())

Pedestrian feature shape: (168, 4)
Bicycle feature shape: (168, 4)


,hour_of_day,day_of_week,weekend_flag,pedestrian_density
0,0,Monday,0,0.046343
1,1,Monday,0,0.012269
2,2,Monday,0,0.008296
3,3,Monday,0,0.000000
4,4,Monday,0,0.005817


,hour_of_day,day_of_week,weekend_flag,bicycle_density
0,0,Monday,0,0.013089
1,1,Monday,0,0.000000
2,2,Monday,0,0.005236
3,3,Monday,0,0.005236
4,4,Monday,0,0.020942


### Observation

The final mobility feature datasets now include the engineered density features along with consistent temporal keys. This structure makes them suitable for later merging with crash and weather features in the integration stage.

In [62]:
# -----------------------------
# 9. Merge mobility features
# -----------------------------
mobility_features = pd.merge(
    pedestrian_features,
    bicycle_features,
    on=["hour_of_day", "day_of_week", "weekend_flag"],
    how="inner"
)

print("Final mobility feature shape:", mobility_features.shape)
mobility_features.head()

Final mobility feature shape: (168, 5)


,hour_of_day,day_of_week,weekend_flag,pedestrian_density,bicycle_density
0,0,Monday,0,0.046343,0.013089
1,1,Monday,0,0.012269,0.000000
2,2,Monday,0,0.008296,0.005236
3,3,Monday,0,0.000000,0.005236
4,4,Monday,0,0.005817,0.020942


## Save Feature Outputs

The engineered mobility features are saved for later dataset integration and predictive modelling.

In [63]:
# -----------------------------
# 10. Save feature dataset
# -----------------------------
from pathlib import Path

PROCESSED_DATA_DIR = Path("data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

mobility_feature_file = PROCESSED_DATA_DIR / "mobility_features.csv"

mobility_features.to_csv(mobility_feature_file, index=False)

print("Saved:", mobility_feature_file)

Saved: data/processed/mobility_features.csv


In [64]:
# -----------------------------
# 11. Download feature files (optional)
# -----------------------------
from google.colab import files

files.download(str(mobility_feature_file))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Output

The feature engineering process generated a unified mobility feature dataset:

- mobility_features.csv

This dataset includes:
- hour_of_day  
- day_of_week  
- weekend_flag  
- pedestrian_density  
- bicycle_density  

It represents relative mobility intensity across temporal conditions and is ready for integration with crash and weather datasets.

## Final Summary

This notebook created two key mobility-based features: pedestrian_density and bicycle_density.

These features were derived by aggregating mobility data across hour of day, day of week, and weekend conditions, followed by min-max normalisation to produce relative intensity values.

The pedestrian and bicycle features were then combined into a unified dataset using shared temporal keys. This ensures consistency with the crash feature dataset and prepares the data for integration in the next stage of the project.

The resulting dataset provides a strong foundation for modelling traffic accident risk.